In [ ]:
from dotenv import load_dotenv, find_dotenv

assert load_dotenv(find_dotenv(usecwd=False)), "The .env file was not loaded."

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
import torch

import json

from drn import (
    GLM,
    split_data,
    preprocess_data,
)

from hyperparameter_tuning_objectives import (
    objective_cann,
    objective_mdn,
    objective_ddr,
    objective_drn,
)

from analysis_utils import rank_models_per_seed, get_nll_crps_rmse_ql

In [ ]:
accelerator = "gpu" if torch.cuda.is_available() else "cpu"
print(f"Using {accelerator} for training.")

In [ ]:
DATA_DIR = Path("data/interim/real")
features = pd.read_csv(
    DATA_DIR / "features.csv", na_values=["NA"], keep_default_na=False
)
target = pd.read_csv(DATA_DIR / "target.csv")

PLOT_DIR = Path("plots/splitting-test")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
glm_gamma_dict_nll = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_gamma_dict_crps = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_gamma_dict_rmse = {"1000": [], "3000": [], "6000": [], "20000": []}
glm_gamma_dict_ql90 = {"1000": [], "3000": [], "6000": [], "20000": []}

drn_gamma_dict_nll = {"1000": [], "3000": [], "6000": [], "20000": []}
mdn_dict_nll = {"1000": [], "3000": [], "6000": [], "20000": []}
ddr_dict_nll = {"1000": [], "3000": [], "6000": [], "20000": []}

drn_gamma_dict_crps = {"1000": [], "3000": [], "6000": [], "20000": []}
mdn_dict_crps = {"1000": [], "3000": [], "6000": [], "20000": []}
ddr_dict_crps = {"1000": [], "3000": [], "6000": [], "20000": []}

drn_gamma_dict_rmse = {"1000": [], "3000": [], "6000": [], "20000": []}
mdn_dict_rmse = {"1000": [], "3000": [], "6000": [], "20000": []}
ddr_dict_rmse = {"1000": [], "3000": [], "6000": [], "20000": []}

drn_gamma_dict_ql90 = {"1000": [], "3000": [], "6000": [], "20000": []}
mdn_dict_ql90 = {"1000": [], "3000": [], "6000": [], "20000": []}
ddr_dict_ql90 = {"1000": [], "3000": [], "6000": [], "20000": []}

ddr_raw_dict_nll = {"1000": [], "3000": [], "6000": [], "20000": []}
ddr_raw_dict_crps = {"1000": [], "3000": [], "6000": [], "20000": []}
ddr_raw_dict_rmse = {"1000": [], "3000": [], "6000": [], "20000": []}
ddr_raw_dict_ql90 = {"1000": [], "3000": [], "6000": [], "20000": []}

cann_gamma_dict_nll = {"1000": [], "3000": [], "6000": [], "20000": []}
cann_gamma_dict_crps = {"1000": [], "3000": [], "6000": [], "20000": []}
cann_gamma_dict_rmse = {"1000": [], "3000": [], "6000": [], "20000": []}
cann_gamma_dict_ql90 = {"1000": [], "3000": [], "6000": [], "20000": []}

# Training

In [ ]:
cat_features = [
    "HasKmLimit",
    "Garage",
    "Gender",
    "MariStat",
    "SocioCateg",
    "VehBody",
    "VehEngine",
    "VehEnergy",
    "VehClass",
    "VehPrice",
    "VehUsage",
]

num_features = [feature for feature in features.columns if feature not in cat_features]

In [ ]:
np.random.seed(31052025)

NUM_RANDOM_SPLITS = 100
split_seeds = [int(s) for s in np.random.randint(0, 2**32 - 1, size=NUM_RANDOM_SPLITS)]

distribution = "gamma"

batch_size = 128
patience = 50
lr = 1e-3

hidden_size = 256
dropout_rate = 0.2
num_hidden_layers = 3

num_components = 5

proportion = 0.25

min_obs = 3
kl_alpha = 5e-3
mean_alpha = 5e-2
dv_alpha = 5e-2
kl_direction = "forwards"
criteria = "CRPS"


for split_seed in split_seeds:
    print(f"Seed: {split_seed}")
    print(
        "-----------------------------------------------------------------------------------"
    )
    x_train_raw, x_val_raw, x_test_raw, y_train, y_val, y_test = split_data(
        features, target, seed=split_seed
    )

    x_train, x_val, x_test, ct, _ = preprocess_data(
        x_train_raw,
        x_val_raw,
        x_test_raw,
        num_features=num_features,
        cat_features=cat_features,
        num_standard=True,
    )

    X_train = torch.Tensor(x_train.values)
    X_val = torch.Tensor(x_val.values)
    X_test = torch.Tensor(x_test.values)
    Y_train = torch.Tensor(y_train.values).flatten()
    Y_val = torch.Tensor(y_val.values).flatten()
    Y_test = torch.Tensor(y_test.values).flatten()

    glm = GLM.from_statsmodels(X_train, Y_train, distribution=distribution)

    _, cann = objective_cann(
        num_hidden_layers,
        hidden_size,
        dropout_rate,
        lr,
        batch_size,
        glm,
        X_train,
        Y_train,
        X_val,
        Y_val,
        accelerator,
        patience,
    )

    _, mdn = objective_mdn(
        num_hidden_layers,
        hidden_size,
        dropout_rate,
        lr,
        num_components,
        batch_size,
        distribution,
        X_train,
        Y_train,
        X_val,
        Y_val,
        accelerator,
        patience,
    )

    _, ddr = objective_ddr(
        num_hidden_layers,
        hidden_size,
        dropout_rate,
        lr,
        proportion,
        batch_size,
        X_train,
        Y_train,
        X_val,
        Y_val,
        accelerator,
        patience,
    )

    _, drn = objective_drn(
        num_hidden_layers,
        hidden_size,
        dropout_rate,
        lr,
        kl_alpha,
        mean_alpha,
        dv_alpha,
        batch_size,
        proportion,
        min_obs,
        glm,
        kl_direction,
        criteria,
        X_train,
        Y_train,
        X_val,
        Y_val,
        accelerator,
        patience,
    )

    nll_test_dict, crps_test_dict, rmse_test_dict, ql_90_test_dict = (
        get_nll_crps_rmse_ql(
            models=[drn, cann, mdn, ddr, glm],
            names=["DRN", "CANN", "MDN", "DDR", "GLM"],
            X_test_data=X_test,
            Y_test_data=Y_test,
            y_train=y_train,
        )
    )

    size = 1_000
    drn_gamma_dict_nll[f"{size}"].append(nll_test_dict["DRN"].item())
    drn_gamma_dict_crps[f"{size}"].append(crps_test_dict["DRN"].item())
    drn_gamma_dict_rmse[f"{size}"].append(rmse_test_dict["DRN"].item())
    drn_gamma_dict_ql90[f"{size}"].append(ql_90_test_dict["DRN"].item())

    cann_gamma_dict_nll[f"{size}"].append(nll_test_dict["CANN"].item())
    cann_gamma_dict_crps[f"{size}"].append(crps_test_dict["CANN"].item())
    cann_gamma_dict_rmse[f"{size}"].append(rmse_test_dict["CANN"].item())
    cann_gamma_dict_ql90[f"{size}"].append(ql_90_test_dict["CANN"].item())

    ddr_raw_dict_nll[f"{size}"].append(nll_test_dict["DDR"].item())
    ddr_raw_dict_crps[f"{size}"].append(crps_test_dict["DDR"].item())
    ddr_raw_dict_rmse[f"{size}"].append(rmse_test_dict["DDR"].item())
    ddr_raw_dict_ql90[f"{size}"].append(ql_90_test_dict["DDR"].item())

    mdn_dict_nll[f"{size}"].append(nll_test_dict["MDN"].item())
    mdn_dict_crps[f"{size}"].append(crps_test_dict["MDN"].item())
    mdn_dict_rmse[f"{size}"].append(rmse_test_dict["MDN"].item())
    mdn_dict_ql90[f"{size}"].append(ql_90_test_dict["MDN"].item())

    glm_gamma_dict_nll[f"{size}"].append(nll_test_dict["GLM"].item())
    glm_gamma_dict_crps[f"{size}"].append(crps_test_dict["GLM"].item())
    glm_gamma_dict_rmse[f"{size}"].append(rmse_test_dict["GLM"].item())
    glm_gamma_dict_ql90[f"{size}"].append(ql_90_test_dict["GLM"].item())

In [ ]:
# Combine all dictionaries
data_dicts = {
    "glm_gamma_dict_nll": glm_gamma_dict_nll,
    "glm_gamma_dict_crps": glm_gamma_dict_crps,
    "glm_gamma_dict_rmse": glm_gamma_dict_rmse,
    "glm_gamma_dict_ql90": glm_gamma_dict_ql90,
    "drn_gamma_dict_nll": drn_gamma_dict_nll,
    "mdn_dict_nll": mdn_dict_nll,
    "ddr_dict_nll": ddr_dict_nll,
    "drn_gamma_dict_crps": drn_gamma_dict_crps,
    "mdn_dict_crps": mdn_dict_crps,
    "ddr_dict_crps": ddr_dict_crps,
    "drn_gamma_dict_rmse": drn_gamma_dict_rmse,
    "mdn_dict_rmse": mdn_dict_rmse,
    "ddr_dict_rmse": ddr_dict_rmse,
    "drn_gamma_dict_ql90": drn_gamma_dict_ql90,
    "mdn_dict_ql90": mdn_dict_ql90,
    "ddr_dict_ql90": ddr_dict_ql90,
    "ddr_raw_dict_nll": ddr_raw_dict_nll,
    "ddr_raw_dict_crps": ddr_raw_dict_crps,
    "ddr_raw_dict_rmse": ddr_raw_dict_rmse,
    "ddr_raw_dict_ql90": ddr_raw_dict_ql90,
    "cann_gamma_dict_nll": cann_gamma_dict_nll,
    "cann_gamma_dict_crps": cann_gamma_dict_crps,
    "cann_gamma_dict_rmse": cann_gamma_dict_rmse,
    "cann_gamma_dict_ql90": cann_gamma_dict_ql90,
}

# Save the data
with open(PLOT_DIR / "rank_distribution_data.json", "w") as f:
    json.dump(data_dicts, f, indent=4)

# Visualisation

In [ ]:
# Load the dictionary back from the file
with open(PLOT_DIR / "rank_distribution_data.json", "r") as f:
    loaded_dicts = json.load(f)

# Dynamically assign all loaded dictionaries as variables
globals().update(loaded_dicts)

In [ ]:
# Dicts for each metric
metric_dicts_nll = {
    "DRN": drn_gamma_dict_nll,
    "CANN": cann_gamma_dict_nll,
    "MDN": mdn_dict_nll,
    "DDR": ddr_raw_dict_nll,
    "GLM": glm_gamma_dict_nll,
}

metric_dicts_crps = {
    "DRN": drn_gamma_dict_crps,
    "CANN": cann_gamma_dict_crps,
    "MDN": mdn_dict_crps,
    "DDR": ddr_raw_dict_crps,
    "GLM": glm_gamma_dict_crps,
}

metric_dicts_rmse = {
    "DRN": drn_gamma_dict_rmse,
    "CANN": cann_gamma_dict_rmse,
    "MDN": mdn_dict_rmse,
    "DDR": ddr_raw_dict_rmse,
    "GLM": glm_gamma_dict_rmse,
}

metric_dicts_ql90 = {
    "DRN": drn_gamma_dict_ql90,
    "CANN": cann_gamma_dict_ql90,
    "MDN": mdn_dict_ql90,
    "DDR": ddr_raw_dict_ql90,
    "GLM": glm_gamma_dict_ql90,
}

# List of models
models = ["GLM", "CANN", "MDN", "DDR", "DRN"]

# Get rankings for size = 1000 (change if needed)
ranks_nll = rank_models_per_seed(metric_dicts_nll, "nll", models)
ranks_crps = rank_models_per_seed(metric_dicts_crps, "crps", models)
ranks_rmse = rank_models_per_seed(metric_dicts_rmse, "rmse", models)
ranks_ql90 = rank_models_per_seed(metric_dicts_ql90, "ql90", models)


# Redefine the metric_ranks dictionary from the earlier context
metric_ranks = {
    "NLL": ranks_nll,
    "CRPS": ranks_crps,
    "RMSE": ranks_rmse,
    "QL90": ranks_ql90,
}

In [ ]:
# Plotting side-by-side histograms at each x-tick (bar plot style)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

rank_bins = [1, 2, 3]  # 4, 5]
bar_width = 0.15
model_list = list(ranks_nll.columns)
x = np.arange(len(rank_bins))

for ax, (metric, rank_df) in zip(axes, metric_ranks.items()):
    for i, model in enumerate(model_list):
        counts = (
            rank_df[model].value_counts().reindex(rank_bins, fill_value=0).sort_index()
        )
        ax.bar(x + i * bar_width, counts.values, width=bar_width, label=model)

    ax.set_title(f"{metric} Rank Distribution")
    ax.set_xlabel("Rank (1 = Best)")
    ax.set_ylabel("Frequency")
    ax.set_xticks(x + bar_width * 2)
    ax.set_xticklabels(rank_bins)
    ax.legend()
    ax.grid(True, axis="y")

plt.tight_layout()
plt.savefig(PLOT_DIR / "rank_distributions.png")

In [ ]:
# Collect stats for all metrics
summary_stats = []

for metric_name, df in metric_ranks.items():
    for model in df.columns:
        summary_stats.append(
            {
                "Metric": metric_name,
                "Model": model,
                "Mean": df[model].mean(),
                "Std": df[model].std(),
            }
        )

summary_df = pd.DataFrame(summary_stats)

# Unique models and metrics for plotting
models = summary_df["Model"].unique()
metrics = summary_df["Metric"].unique()
colors = dict(zip(models, sns.color_palette(n_colors=len(models))))

# Manually assign colors to each model
colors = {
    "DRN": "blue",
    "CANN": "teal",
    "MDN": "orange",
    "DDR": "black",
    "GLM": "olive",
}

# Regenerate the plot with larger title and legend font sizes
plt.figure(figsize=(10, 5), dpi=200)  # High resolution

offset = 0.1  # horizontal spacing between model points
for i, metric in enumerate(metrics):
    for j, model in enumerate(models):
        row = summary_df[
            (summary_df["Metric"] == metric) & (summary_df["Model"] == model)
        ].iloc[0]
        x = i + (j - len(models) / 2) * offset
        color = colors[model]
        plt.plot(
            [x, x],
            [row["Mean"] - row["Std"], row["Mean"] + row["Std"]],
            color=color,
            linewidth=3,
        )
        plt.plot(x, row["Mean"], "o", color=color, label=model, markersize=12)

# Deduplicate legend
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(
    by_label.values(),
    by_label.keys(),
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    fontsize=18,
)

plt.xticks(ticks=range(len(metrics)), labels=metrics, fontsize=15)
plt.yticks(fontsize=15)
plt.title("Model Ranking Across Evaluation Metrics", fontsize=18)
plt.ylabel("Rank (1 = Best)", fontsize=15)
plt.ylim(1, 5.2)
plt.tight_layout()
plt.savefig(PLOT_DIR / "model_ranking_summary.png")